In [1]:
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import DataLoader

from torchvision import datasets, models
from torchvision.transforms import transforms

from sklearn.metrics import auc

In [ ]:
def get_device():
    return torch.device("cuda")


def load_model(model_path, num_classes=10):
    device = get_device()

    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)

    state_dict = torch.load(Path(model_path), map_location=device)
    model.load_state_dict(state_dict)

    model = model.to(device)
    model.eval()

    return model


def get_test_data():
    transform = transforms.Compose(
        [
            transforms.Resize((224, 224)),
            transforms.Grayscale(num_output_channels=3),
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,)),
        ]
    )

    return datasets.MNIST(
        root="../data", download=False, train=False, transform=transform
    )


def get_test_data_loader():
    return DataLoader(
        get_test_data(),
        batch_size=64,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
    )


def tensor_to_img(tensor):
    # Remove batch dimension (if exists) and move to CPU
    img_np = tensor.squeeze(0).detach().cpu().numpy()

    # Handle channel dimensions (C, H, W) -> (H, W, C)
    if img_np.ndim == 3:
        if img_np.shape[0] == 1:
            img_np = img_np.squeeze(0)  # Grayscale conversion
        else:
            img_np = np.transpose(img_np, (1, 2, 0))  # RGB conversion

    # Denormalize from [-1, 1] to [0, 1] for visual display
    img_np = (img_np + 1) / 2
    img_np = np.clip(img_np, 0, 1)

    return img_np


def add_gaussian_noise(tensor, sigma=0.1):
    noise = torch.randn_like(tensor) * sigma
    noisy_tensor = tensor + noise

    return torch.clamp(noisy_tensor, -1.0, 1.0)


def process_attribution_raw(attr_tensor):
    attr_np = attr_tensor.squeeze(0).detach().cpu().numpy()

    if attr_np.ndim == 3:
        attr_np = np.sum(attr_np, axis=0)

    return attr_np


def plot_image(ax, img, title, cmap=None):
    ax.imshow(img, cmap=cmap)
    ax.set_title(title)
    ax.axis("off")


def cosine_sim(a, b):
    a = a.flatten()
    b = b.flatten()

    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)


def topk_iou(a, b, top_k_percent=0.10):
    a = a.flatten()
    b = b.flatten()

    k = int(len(a) * top_k_percent)

    # Top-K największych wartości RAW attribution
    top_a = np.argpartition(a, -k)[-k:]
    top_b = np.argpartition(b, -k)[-k:]

    set_a = set(top_a)
    set_b = set(top_b)

    intersection = len(set_a & set_b)
    union = len(set_a | set_b)

    return intersection / (union + 1e-8)


def deletion_auc(
    model,
    input_tensor,
    attribution_map,
    target_class,
    steps=20,
    baseline_value=-1.0,
):
    """
    Deletion AUC dla jednej próbki.

    Usuwa piksele według największych DODATNICH atrybucji,
    czyli tych, które wspierają target_class.
    """

    x_deleted = input_tensor.clone().detach()
    _, C, H, W = x_deleted.shape

    total_pixels = H * W

    # RAW attribution -> flatten
    attr_flat = attribution_map.flatten()

    # Bierzemy tylko dodatnie atrybucje jako importance
    # bo deletion ma usuwać piksele wspierające target class
    importance = np.maximum(attr_flat, 0)

    # Jeśli przypadkiem wszystko byłoby zerowe lub ujemne,
    # fallback na surowe wartości dodatnie nie zadziała, więc używamy abs awaryjnie
    if np.all(importance == 0):
        importance = np.abs(attr_flat)

    # Indeksy od najbardziej do najmniej ważnych
    sorted_indices = np.argsort(importance)[::-1]

    confidences = []
    fractions_removed = []

    with torch.no_grad():
        for step in range(steps + 1):
            fraction_removed = step / steps

            # Mierzymy confidence aktualnego obrazu
            output = model(x_deleted)
            prob = torch.softmax(output, dim=1)[0, target_class].item()

            confidences.append(prob)
            fractions_removed.append(fraction_removed)

            if step == steps:
                break

            start = int((step / steps) * total_pixels)
            end = int(((step + 1) / steps) * total_pixels)

            pixels_to_remove = sorted_indices[start:end]

            rows = pixels_to_remove // W
            cols = pixels_to_remove % W

            # Usuwamy piksele we wszystkich kanałach
            x_deleted[:, :, rows, cols] = baseline_value

    deletion_score = auc(fractions_removed, confidences)

    return deletion_score